# 🔬 ThyroScan Pro V25 — Ultimate Master Engine

Este notebook integra o treinamento do modelo de **Machine Learning Clássico** (sem Deep Learning) com o **Frontend HTML Interativo**, permitindo que você avalie o modelo e faça predições em tempo real no Google Colab.

**Instruções:**
1. Rode a célula de Instalação e Treinamento. Será solicitado que você faça o **upload do arquivo `.zip`** contendo as pastas `benign` e `malignant`.
2. Após o treinamento, o modelo será salvo no Colab (`thyroscan_v25_model.pkl`).
3. Rode a última célula para carregar a **Interface HTML**. Você poderá fazer upload de novas imagens na interface e o modelo python responderá em tempo real!

### 1. Treinamento do Modelo (Engine V25)

In [ ]:
!pip install imbalanced-learn scikit-image

# Importa e executa o pipeline principal que criamos no arquivo thyroscan_v25.py
import thyroscan_v25

print("\nIniciando o treinamento. Por favor, faça o upload do seu dataset em formato .zip quando solicitado.\n")
thyroscan_v25.main()

### 2. Interface Interativa HTML (Frontend)

In [ ]:
import IPython
from google.colab import output
import cv2
import numpy as np
import base64
import os
from thyroscan_v25 import ThyroInference

# Verifica se o modelo foi gerado corretamente
if not os.path.exists('thyroscan_v25_model.pkl'):
    print("❌ Erro: O modelo não foi encontrado. Por favor, rode a célula de treinamento primeiro.")
else:
    print("✅ Modelo encontrado! Inicializando o motor de inferência...")
    # Carrega o modelo treinado
    inference_engine = ThyroInference('thyroscan_v25_model.pkl')

    # Função de ponte entre o JavaScript (HTML) e o Python
    def thyroscan_classificar(imagem_base64):
        try:
            # Remover o prefixo do Base64 vindo do HTML
            if ',' in imagem_base64:
                imagem_base64 = imagem_base64.split(',')[1]

            # Decodificar Base64 para array numpy e ler via OpenCV
            img_bytes = base64.b64decode(imagem_base64)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            img_cv2 = cv2.imdecode(img_np, cv2.IMREAD_GRAYSCALE)

            if img_cv2 is None:
                return IPython.display.JSON({"error": "Imagem inválida ou corrompida.", "simulado": True})

            # Realizar a predição via modelo
            classe, prob, roi = inference_engine.classificar_array(img_cv2)

            # Codificar a ROI (Nódulo isolado) para mostrar no HTML
            _, roi_encoded = cv2.imencode('.png', roi)
            roi_base64 = base64.b64encode(roi_encoded).decode('utf-8')

            return IPython.display.JSON({
                "classe": classe,
                "probabilidade": float(prob),
                "threshold": float(inference_engine.threshold),
                "roi_base64": roi_base64,
                "simulado": False
            })
        except Exception as e:
            print(f"Erro na inferência: {e}")
            return IPython.display.JSON({"error": str(e), "simulado": True})

    # Registra a função no Google Colab para ser chamada pelo frontend JS
    output.register_callback('thyroscan_classificar', thyroscan_classificar)
    print("✅ Ponte JS-Python estabelecida com sucesso.")

    # Lê e exibe o Frontend HTML
    if os.path.exists('thyroscan_frontend.html'):
        with open('thyroscan_frontend.html', 'r', encoding='utf-8') as f:
            html_code = f.read()
        # Exibe o HTML com altura de 900px para acomodar bem o design no Colab
        display(IPython.display.HTML(html_code))
    else:
        print("❌ Erro: O arquivo thyroscan_frontend.html não foi encontrado no diretório atual.")
